# Finalized Strategic Trend Analysis Pipeline (2026)
This consolidated script handles semantic expansion, trend forecasting via Facebook Prophet, and automated PDF reporting.

In [ ]:
import pandas as pd
import time
import requests
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
from pytrends.request import TrendReq
from prophet import Prophet
from scipy.signal import find_peaks
from sklearn.cluster import KMeans
from matplotlib.backends.backend_pdf import PdfPages
import textwrap

# --- CONFIGURATION ---
BASE_KEYWORDS = ['war']
GEO = 'RU'
TIMEFRAME = 'today 5-y'
HL = 'ru-RU'
YEAR_TO_FORECAST = 2026
AUTHOR_NAME = "Eduard Samokhvalov"
AUTHOR_TITLE = "Quantitative Developer"
CONTACT_INFO = "edward.samokhvalov@gmail.com | Telegram: @EduardSam"

# Initialize Trends with a session to handle potential protocol issues
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

session = requests.Session()
retry = Retry(connect=5, backoff_factor=1, allowed_methods=None)
adapter = HTTPAdapter(max_retries=retry)
session.mount('https://', adapter)

# Set timeout directly in TrendReq to avoid double argument error in requests_args
pytrends = TrendReq(hl=HL, tz=360, timeout=30, requests_args={'verify':True})

def expand_keywords(base_list):
    expanded = set(base_list)
    for word in base_list:
        try:
            print(f"Expanding: {word}...")
            response = requests.get(f'https://api.datamuse.com/words?rel_trg={word}&max=5', timeout=10)
            if response.status_code == 200:
                for res in response.json():
                    expanded.add(res['word'])
            time.sleep(random.uniform(3, 5))
            suggestions = pytrends.suggestions(word)
            for s in suggestions[:3]:
                expanded.add(s['title'])
            time.sleep(random.uniform(5, 8))
        except Exception as e:
            print(f"Expansion error for '{word}': {e}")
    return list(expanded)

full_keywords = expand_keywords(BASE_KEYWORDS)
print(f"Semantic Core: {full_keywords}")

all_yearly_seasonalities = {}
all_individual_peak_dates = []

for kw in full_keywords:
    try:
        print(f"Processing: {kw}...")
        time.sleep(random.uniform(25, 35))

        pytrends.build_payload([kw], timeframe=TIMEFRAME, geo=GEO)
        df_trends = pytrends.interest_over_time()

        if df_trends.empty or kw not in df_trends.columns:
            print(f"No data found for {kw}, skipping.")
            continue

        df_p = df_trends.reset_index()[['date', kw]].rename(columns={'date': 'ds', kw: 'y'})
        m = Prophet(seasonality_mode='multiplicative', yearly_seasonality=True)
        m.fit(df_p)

        future = m.make_future_dataframe(periods=450, freq='D')
        forecast = m.predict(future)
        f_2026 = forecast[forecast['ds'].dt.year == YEAR_TO_FORECAST]

        dates = f_2026['ds'].values
        yearly = f_2026['yearly'].values
        norm_yearly = yearly / np.abs(yearly).max() if np.abs(yearly).max() > 0 else yearly
        all_yearly_seasonalities[kw] = (dates, norm_yearly)

        p_idx, _ = find_peaks(norm_yearly, prominence=0.1)
        for idx in p_idx:
            all_individual_peak_dates.append({'date': pd.to_datetime(dates[idx]), 'keyword': kw})

    except Exception as e:
        print(f"Error processing {kw}: {e}")
        time.sleep(60)

if all_yearly_seasonalities:
    seasonality_matrix = np.array([v[1] for v in all_yearly_seasonalities.values()])
    n_clusters = min(3, len(all_yearly_seasonalities))
    km = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = km.fit_predict(seasonality_matrix)

    pdf_path = '/content/Final_Strategic_Analysis_2026.pdf'
    with PdfPages(pdf_path) as pdf:
        fig_t = plt.figure(figsize=(8.5, 11))
        plt.text(0.5, 0.7, 'Strategic Trend Forecast 2026', fontsize=22, ha='center', fontweight='bold')
        plt.text(0.5, 0.6, f'Domain: {BASE_KEYWORDS[0].upper()}', fontsize=16, ha='center')
        plt.text(0.5, 0.4, f'Prepared by: {AUTHOR_NAME}', fontsize=12, ha='center')
        plt.axis('off')
        pdf.savefig(fig_t)
        plt.close()

        for i in range(n_clusters):
            fig_c, ax = plt.subplots(figsize=(12, 6))
            k_names = [list(all_yearly_seasonalities.keys())[j] for j, l in enumerate(labels) if l == i]
            for kn in k_names:
                ax.plot(all_yearly_seasonalities[kn][0], all_yearly_seasonalities[kn][1], label=kn, alpha=0.7)
            ax.set_title(f'Cluster {i+1} Seasonality Patterns')
            ax.legend()
            pdf.savefig(fig_c)
            plt.close()
    print(f"Final Report saved to: {pdf_path}")
else:
    print("Analysis failed: No data retrieved from Google Trends. Please try again later.")

Expanding: war...
Expansion error for 'war': The request failed: Google returned a response with code 429
Semantic Core: ['war', 'veterans', 'crimes', 'fought', 'succession', 'confederate']
Processing: war...
Error processing war: The request failed: Google returned a response with code 429
Processing: veterans...
Error processing veterans: The request failed: Google returned a response with code 429
Processing: crimes...
Error processing crimes: The request failed: Google returned a response with code 429
Processing: fought...
Error processing fought: The request failed: Google returned a response with code 429
Processing: succession...
Error processing succession: The request failed: Google returned a response with code 429
Processing: confederate...
Error processing confederate: The request failed: Google returned a response with code 429
